# Veri Ön İşleme Pipeline — Student Attention Detection

Bu notebook, **TÜBİTAK 2209-B Öğrenci Dikkat Tespiti** projesi kapsamında toplanan
~187K yüz ifadesi görüntüsünü temizleyip eğitim için hazırlar.

## Pipeline Adımları
1. **Bozuk görüntü tespiti** — okunamayan, çok küçük ve düşük varyanslı dosyalar
2. **Kopya tespit** — pHash ile tam ve yakın kopyalar, split-leak kontrolü
3. **FER+ etiket analizi** — crowd-vote güven skorları ve 3-sınıf eşleştirmesi
4. **Stratified bölümleme** — %80 train / %10 val / %10 test
5. **Augmentation önizleme** — Phase 1 (light), Phase 2 (strong), MixUp/CutMix, normalizasyon

## Kaynak ve Çıktı
| Klasör | Açıklama |
|---|---|
| `data/processed_merged/` | Birleştirilmiş kaynak veri (10 dataset) |
| `data/cleaned/` | Temizlenmiş ve bölümlenmiş veri (train/val/test) |
| `data/quarantine/` | Kaldırılan görüntüler |
| `results/data_cleaning/` | Raporlar ve grafikler |

In [ ]:
# Install required packages
!pip install imagehash opencv-python-headless pandas matplotlib seaborn scikit-learn tqdm albumentations Pillow -q

In [ ]:
# Mount Google Drive and define paths
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/Tubitak-2209B"
DATA_DIR = f"{DRIVE_ROOT}/data/processed_merged"
CLEANED_DIR = f"{DRIVE_ROOT}/data/cleaned"
QUARANTINE_DIR = f"{DRIVE_ROOT}/data/quarantine"
REPORTS_DIR = f"{DRIVE_ROOT}/results/data_cleaning"

import os
for d in [CLEANED_DIR, QUARANTINE_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

In [ ]:
# Check for tar.gz archive and extract if data directory is missing
import tarfile
from pathlib import Path

data_path = Path(DATA_DIR)
if not data_path.exists():
    tar_path = Path(f"{DRIVE_ROOT}/data/processed_merged.tar.gz")
    if tar_path.exists():
        print(f"Extracting {tar_path}...")
        with tarfile.open(tar_path, "r:gz") as tar:
            tar.extractall(path=str(tar_path.parent))
        print("Extraction complete.")
    else:
        raise FileNotFoundError(f"Data not found: {DATA_DIR}")
else:
    print(f"Data directory found: {DATA_DIR}")

In [ ]:
# ============================================================
# Config constants (self-contained, no project imports)
# ============================================================

# Image pipeline
IMAGE_SIZE = 224
CLASS_NAMES = ["negative", "neutral", "positive"]
NUM_CLASSES = 3

# Split ratios (merged dataset)
MERGED_TEST_RATIO = 0.10
MERGED_VAL_RATIO = 0.10
RANDOM_SEED = 42

# pHash duplicate detection
PHASH_HASH_SIZE = 16
PHASH_HAMMING_THRESHOLD = 6
ENABLE_NEAR_DUP_SCAN = True
MAX_NEAR_DUP_CANDIDATES_PER_IMAGE = 4000

# Corrupt image thresholds
MIN_IMAGE_SIZE = 10        # pixels
MIN_FILE_SIZE_BYTES = 100  # bytes

# FER+ relabeling
FERPLUS_CONFIDENCE_THRESHOLD = 0.4

# ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# MixUp / CutMix
MIXUP_ALPHA = 0.2
CUTMIX_ALPHA = 1.0
MIXUP_CUTMIX_PROB = 0.5

# Dataset source prefixes
_DATASET_PREFIXES = {
    "fer2013img_": "FER2013",
    "faceexpr_": "Face-Expression",
    "rafdb_": "RAF-DB",
    "affyolo_": "AffectNet-YOLO",
    "afftrain_": "AffectNet-Train",
    "expw_": "ExpW",
    "ckplus_": "CK+",
    "kdef_": "KDEF",
}

# FER+ emotion mapping
_FERPLUS_TO_3CLASS = {
    "neutral": "neutral",
    "happiness": "positive",
    "surprise": None,
    "sadness": "negative",
    "anger": "negative",
    "disgust": "negative",
    "fear": "negative",
    "contempt": "negative",
}
_FERPLUS_EMOTION_COLS = [
    "neutral", "happiness", "surprise", "sadness",
    "anger", "disgust", "fear", "contempt",
]

# Image file extensions
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}

print("Config constants loaded:")
print(f"  IMAGE_SIZE             = {IMAGE_SIZE}")
print(f"  CLASS_NAMES            = {CLASS_NAMES}")
print(f"  MERGED_TEST_RATIO      = {MERGED_TEST_RATIO}")
print(f"  MERGED_VAL_RATIO       = {MERGED_VAL_RATIO}")
print(f"  RANDOM_SEED            = {RANDOM_SEED}")
print(f"  PHASH_HASH_SIZE        = {PHASH_HASH_SIZE}")
print(f"  PHASH_HAMMING_THRESHOLD= {PHASH_HAMMING_THRESHOLD}")
print(f"  ENABLE_NEAR_DUP_SCAN   = {ENABLE_NEAR_DUP_SCAN}")
print(f"  MAX_NEAR_DUP_CANDIDATES= {MAX_NEAR_DUP_CANDIDATES_PER_IMAGE}")
print(f"  MIN_IMAGE_SIZE         = {MIN_IMAGE_SIZE}")
print(f"  MIN_FILE_SIZE_BYTES    = {MIN_FILE_SIZE_BYTES}")
print(f"  FERPLUS_CONFIDENCE_THR = {FERPLUS_CONFIDENCE_THRESHOLD}")
print(f"  IMAGENET_MEAN          = {IMAGENET_MEAN}")
print(f"  IMAGENET_STD           = {IMAGENET_STD}")
print(f"  MIXUP_ALPHA            = {MIXUP_ALPHA}")
print(f"  CUTMIX_ALPHA           = {CUTMIX_ALPHA}")
print(f"  Dataset prefixes       = {list(_DATASET_PREFIXES.values())}")

In [ ]:
# Initial data count: split x class table
import pandas as pd
from pathlib import Path

data_root = Path(DATA_DIR)
splits = ["train", "val", "test"]
rows = []

for split in splits:
    split_dir = data_root / split
    if not split_dir.exists():
        rows.append({"split": split, **{c: 0 for c in CLASS_NAMES}, "total": 0})
        continue
    counts = {}
    for cls in CLASS_NAMES:
        cls_dir = split_dir / cls
        if cls_dir.exists():
            counts[cls] = sum(
                1 for f in cls_dir.iterdir()
                if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
            )
        else:
            counts[cls] = 0
    rows.append({"split": split, **counts, "total": sum(counts.values())})

# Add totals row
totals = {"split": "TOTAL"}
for cls in CLASS_NAMES:
    totals[cls] = sum(r[cls] for r in rows)
totals["total"] = sum(r["total"] for r in rows)
rows.append(totals)

df_counts = pd.DataFrame(rows).set_index("split")
print("\n=== Initial Data Counts ===")
display(df_counts.style.format("{:,}").set_caption("Processed Merged Dataset"))

In [ ]:
# Dataset report: class distribution bar chart, source pie chart, size stats
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

# Count files by split and class
split_class_counts = {}
all_filenames = []

for split in ["train", "val", "test"]:
    split_dir = data_root / split
    if not split_dir.exists():
        continue
    split_class_counts[split] = {}
    for cls in CLASS_NAMES:
        cls_dir = split_dir / cls
        if cls_dir.exists():
            files = [
                f for f in cls_dir.iterdir()
                if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
            ]
            split_class_counts[split][cls] = len(files)
            all_filenames.extend([f.name for f in files])
        else:
            split_class_counts[split][cls] = 0

# Identify source dataset from filename prefix
source_counts = Counter()
for fname in all_filenames:
    matched = False
    for prefix, dataset_name in _DATASET_PREFIXES.items():
        if fname.startswith(prefix):
            source_counts[dataset_name] += 1
            matched = True
            break
    if not matched:
        source_counts["Other"] += 1

# Create figure with 2 subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart: class distribution grouped by split
x = np.arange(len(CLASS_NAMES))
width = 0.25
for i, split in enumerate(split_class_counts):
    values = [split_class_counts[split].get(c, 0) for c in CLASS_NAMES]
    ax1.bar(x + i * width, values, width, label=split)

ax1.set_xlabel("Class")
ax1.set_ylabel("Count")
ax1.set_title("Class Distribution by Split")
ax1.set_xticks(x + width)
ax1.set_xticklabels(CLASS_NAMES)
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

# Pie chart: source datasets
labels = list(source_counts.keys())
sizes = list(source_counts.values())
ax2.pie(sizes, labels=labels, autopct="%1.1f%%", startangle=90)
ax2.set_title("Source Dataset Distribution")

plt.tight_layout()
plt.savefig(f"{REPORTS_DIR}/dataset_overview.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nTotal images: {len(all_filenames):,}")
print(f"Source datasets: {len(source_counts)}")
for name, count in source_counts.most_common():
    print(f"  {name}: {count:,} ({count/len(all_filenames)*100:.1f}%)")

In [ ]:
# Sample image grid: 3 classes x 5 examples
import cv2
import random

random.seed(RANDOM_SEED)

fig, axes = plt.subplots(3, 5, figsize=(15, 9))
fig.suptitle("Sample Images per Class (train split)", fontsize=14)

for row_idx, cls in enumerate(CLASS_NAMES):
    cls_dir = data_root / "train" / cls
    if not cls_dir.exists():
        for col_idx in range(5):
            axes[row_idx, col_idx].axis("off")
            axes[row_idx, col_idx].set_title(f"{cls} (missing)")
        continue

    all_imgs = [
        f for f in cls_dir.iterdir()
        if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
    ]
    samples = random.sample(all_imgs, min(5, len(all_imgs)))

    for col_idx in range(5):
        ax = axes[row_idx, col_idx]
        if col_idx < len(samples):
            img = cv2.imread(str(samples[col_idx]))
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                ax.imshow(img)
            ax.set_title(f"{cls}", fontsize=10)
        ax.axis("off")

plt.tight_layout()
plt.savefig(f"{REPORTS_DIR}/sample_grid.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Image size distribution histogram
from tqdm.auto import tqdm

# Collect all image paths
all_image_paths = [
    p for p in data_root.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
]

# Sample up to 1000 images for speed
random.seed(RANDOM_SEED)
sample_paths = random.sample(all_image_paths, min(1000, len(all_image_paths)))

widths = []
heights = []
for img_path in tqdm(sample_paths, desc="Reading image sizes"):
    img = cv2.imread(str(img_path))
    if img is not None:
        h, w = img.shape[:2]
        widths.append(w)
        heights.append(h)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(widths, bins=50, alpha=0.7, color="steelblue", edgecolor="black")
ax1.set_title("Width Distribution")
ax1.set_xlabel("Width (px)")
ax1.set_ylabel("Count")
ax1.axvline(np.median(widths), color="red", linestyle="--", label=f"Median: {np.median(widths):.0f}")
ax1.legend()

ax2.hist(heights, bins=50, alpha=0.7, color="coral", edgecolor="black")
ax2.set_title("Height Distribution")
ax2.set_xlabel("Height (px)")
ax2.set_ylabel("Count")
ax2.axvline(np.median(heights), color="red", linestyle="--", label=f"Median: {np.median(heights):.0f}")
ax2.legend()

plt.tight_layout()
plt.savefig(f"{REPORTS_DIR}/size_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n=== Image Size Statistics (n={len(widths)}) ===")
print(f"Width  - min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.1f}, median: {np.median(widths):.0f}")
print(f"Height - min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.1f}, median: {np.median(heights):.0f}")

In [ ]:
# Corrupt image detection (dry-run)
import json

issues = {
    "unreadable": [],
    "too_small_file": [],
    "too_small_image": [],
    "low_variance": [],
}

all_images_for_scan = [
    p for p in data_root.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
]

print(f"Scanning {len(all_images_for_scan)} images for corruption...")

for img_path in tqdm(all_images_for_scan, desc="Corrupt scan"):
    rel_path = str(img_path.relative_to(data_root))

    # File size check
    if img_path.stat().st_size < MIN_FILE_SIZE_BYTES:
        issues["too_small_file"].append(rel_path)
        continue

    # Read check
    img = cv2.imread(str(img_path))
    if img is None:
        issues["unreadable"].append(rel_path)
        continue

    h, w = img.shape[:2]

    # Dimension check
    if h < MIN_IMAGE_SIZE or w < MIN_IMAGE_SIZE:
        issues["too_small_image"].append(rel_path)
        continue

    # Variance check (flat/blank images)
    if np.var(img.astype(np.float32)) < 1.0:
        issues["low_variance"].append(rel_path)
        continue

corrupt_report = {
    "source_dir": str(data_root),
    "total_scanned": len(all_images_for_scan),
    "issues": issues,
    "total_flagged": sum(len(v) for v in issues.values()),
    "action": "dry_run",
}

report_path = Path(REPORTS_DIR) / "corrupt_images.json"
report_path.write_text(json.dumps(corrupt_report, indent=2, ensure_ascii=False))

print(f"\nReport saved to {report_path}")
print("\n--- Corrupt Image Report ---")
print(f"Total scanned : {corrupt_report['total_scanned']:,}")
print(f"Unreadable    : {len(issues['unreadable'])}")
print(f"Too small file: {len(issues['too_small_file'])}")
print(f"Too small img : {len(issues['too_small_image'])}")
print(f"Low variance  : {len(issues['low_variance'])}")
print(f"Total flagged : {corrupt_report['total_flagged']}")

In [ ]:
# Corrupt results visualization
issue_counts = {k: len(v) for k, v in issues.items() if len(v) > 0}

if issue_counts:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Pie chart of issue types
    labels = list(issue_counts.keys())
    sizes = list(issue_counts.values())
    axes[0].pie(sizes, labels=labels, autopct="%1.1f%%", startangle=90,
                colors=["#ff6b6b", "#ffa07a", "#ffd93d", "#c0c0c0"][:len(labels)])
    axes[0].set_title("Corrupt Image Issue Types")

    # Sample grid of corrupt images (up to 2 per category)
    sample_images = []
    sample_labels = []
    for category, paths in issues.items():
        for p in paths[:2]:
            full_path = data_root / p
            img = cv2.imread(str(full_path))
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                sample_images.append(img)
                sample_labels.append(category)

    if sample_images:
        n = min(len(sample_images), 6)
        for i in range(n):
            ax_inner = fig.add_subplot(2, n, n + i + 1)
            ax_inner.imshow(sample_images[i])
            ax_inner.set_title(sample_labels[i], fontsize=8)
            ax_inner.axis("off")

    axes[1].axis("off")
    plt.tight_layout()
    plt.savefig(f"{REPORTS_DIR}/corrupt_visualization.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No corrupt images detected! Dataset is clean.")

print(f"\nTotal flagged: {corrupt_report['total_flagged']} / {corrupt_report['total_scanned']:,}")

In [ ]:
# Duplicate detection (dry-run)
import imagehash
from PIL import Image
from collections import defaultdict

# ---- Helper functions ----

def _hash_to_bits(h):
    """Convert an ImageHash into a deterministic bitstring."""
    return "".join("1" if b else "0" for b in h.hash.flatten())


def _iter_blocks(bit_string, block_count):
    """Split bitstring into almost-equal contiguous blocks."""
    total = len(bit_string)
    block_len = total // block_count
    remainder = total % block_count
    blocks = []
    start = 0
    for idx in range(block_count):
        extra = 1 if idx < remainder else 0
        end = start + block_len + extra
        blocks.append((idx, bit_string[start:end]))
        start = end
    return blocks


def _uf_find(parent, x):
    """Find representative with path compression."""
    if parent[x] != x:
        parent[x] = _uf_find(parent, parent[x])
    return parent[x]


def _uf_union(parent, rank, a, b):
    """Union by rank."""
    ra = _uf_find(parent, a)
    rb = _uf_find(parent, b)
    if ra == rb:
        return
    if rank[ra] < rank[rb]:
        parent[ra] = rb
    elif rank[ra] > rank[rb]:
        parent[rb] = ra
    else:
        parent[rb] = ra
        rank[ra] += 1


# ---- Compute pHash for all images ----
all_images_sorted = sorted(
    p for p in data_root.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)

print(f"Hashing {len(all_images_sorted)} images (hash_size={PHASH_HASH_SIZE})...")

hash_map = defaultdict(list)  # hash_str -> [rel_paths]
file_hashes = {}  # rel_path -> ImageHash

for img_path in tqdm(all_images_sorted, desc="pHash computation"):
    try:
        with Image.open(img_path) as img:
            h = imagehash.phash(img.convert("RGB"), hash_size=PHASH_HASH_SIZE)
        rel = str(img_path.relative_to(data_root))
        hash_map[str(h)].append(rel)
        file_hashes[rel] = h
    except Exception:
        continue

# ---- Exact duplicates ----
exact_groups = {k: v for k, v in hash_map.items() if len(v) > 1}

# ---- Near duplicates (block-based indexing) ----
near_groups = []
candidate_checks = 0

if ENABLE_NEAR_DUP_SCAN:
    rels = sorted(file_hashes.keys())
    n = len(rels)
    hashes_list = [file_hashes[r] for r in rels]
    bit_strings = [_hash_to_bits(h) for h in hashes_list]

    block_count = PHASH_HAMMING_THRESHOLD + 1  # 7 blocks
    block_index = defaultdict(list)

    print("Building block index for near-duplicate scan...")
    for idx, bits in enumerate(bit_strings):
        for block_idx, chunk in _iter_blocks(bits, block_count):
            block_index[(block_idx, chunk)].append(idx)

    parent = list(range(n))
    rank_arr = [0] * n

    print("Scanning near-duplicate candidates...")
    for i in tqdm(range(n), desc="Near-dup scan"):
        candidates = set()
        for block_idx, chunk in _iter_blocks(bit_strings[i], block_count):
            candidates.update(block_index[(block_idx, chunk)])
        candidates.discard(i)

        ordered = [j for j in sorted(candidates) if j > i]
        if MAX_NEAR_DUP_CANDIDATES_PER_IMAGE > 0:
            ordered = ordered[:MAX_NEAR_DUP_CANDIDATES_PER_IMAGE]

        for j in ordered:
            candidate_checks += 1
            dist = hashes_list[i] - hashes_list[j]
            if 0 < dist <= PHASH_HAMMING_THRESHOLD:
                _uf_union(parent, rank_arr, i, j)

    groups = defaultdict(list)
    for idx, rel in enumerate(rels):
        root = _uf_find(parent, idx)
        groups[root].append(rel)

    near_groups = [sorted(group) for group in groups.values() if len(group) > 1]
    near_groups.sort(key=lambda g: g[0])

# ---- Cross-split leak detection ----
all_dup_groups = list(exact_groups.values()) + near_groups
cross_split_leaks = []

for group in all_dup_groups:
    splits_found = defaultdict(list)
    for rel_path in group:
        parts = Path(rel_path).parts
        if parts:
            splits_found[parts[0]].append(rel_path)
    if len(splits_found) > 1:
        cross_split_leaks.append(dict(splits_found))

# ---- Save report ----
dup_report = {
    "source_dir": str(data_root),
    "total_hashed": len(file_hashes),
    "exact_duplicate_groups": len(exact_groups),
    "exact_duplicate_files": sum(len(v) - 1 for v in exact_groups.values()),
    "near_scan_enabled": ENABLE_NEAR_DUP_SCAN,
    "candidate_checks": candidate_checks,
    "near_duplicate_groups": len(near_groups),
    "near_duplicate_files": sum(len(g) - 1 for g in near_groups),
    "cross_split_leaks": cross_split_leaks,
    "action": "dry_run",
    "exact_groups": exact_groups,
    "near_groups": near_groups,
}

dup_report_path = Path(REPORTS_DIR) / "duplicates_report.json"
dup_report_path.write_text(json.dumps(dup_report, indent=2, ensure_ascii=False))

print(f"\nReport saved to {dup_report_path}")
print("\n--- Duplicate Report ---")
print(f"Total hashed        : {dup_report['total_hashed']:,}")
print(f"Exact dup groups    : {dup_report['exact_duplicate_groups']}")
print(f"Exact dup files     : {dup_report['exact_duplicate_files']}")
print(f"Near scan enabled   : {dup_report['near_scan_enabled']}")
print(f"Candidate checks    : {dup_report['candidate_checks']:,}")
print(f"Near dup groups     : {dup_report['near_duplicate_groups']}")
print(f"Near dup files      : {dup_report['near_duplicate_files']}")
print(f"Cross-split leaks   : {len(cross_split_leaks)}")

In [ ]:
# Duplicate results visualization

# Show pairs of duplicate images side by side (up to 5 pairs)
sample_dup_groups = (list(exact_groups.values()) + near_groups)[:5]

if sample_dup_groups:
    n_pairs = len(sample_dup_groups)
    fig, axes = plt.subplots(n_pairs, 2, figsize=(8, 3 * n_pairs))
    if n_pairs == 1:
        axes = [axes]
    fig.suptitle("Duplicate Image Pairs", fontsize=14)

    for row_idx, group in enumerate(sample_dup_groups):
        for col_idx in range(2):
            ax = axes[row_idx][col_idx]
            if col_idx < len(group):
                img_path = data_root / group[col_idx]
                img = cv2.imread(str(img_path))
                if img is not None:
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    ax.imshow(img)
                ax.set_title(Path(group[col_idx]).name, fontsize=7)
            ax.axis("off")

    plt.tight_layout()
    plt.savefig(f"{REPORTS_DIR}/duplicate_pairs.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No duplicate groups found.")

# Cross-split leak summary
print(f"\nCross-split leaks found: {len(cross_split_leaks)}")
for i, leak in enumerate(cross_split_leaks[:5]):
    print(f"  Leak {i+1}: {dict((k, len(v)) for k, v in leak.items())}")

# Bar chart of duplicate counts per split
dup_per_split = defaultdict(int)
for group in all_dup_groups:
    for rel_path in group[1:]:  # skip the first (kept) image
        split_name = Path(rel_path).parts[0] if Path(rel_path).parts else "unknown"
        dup_per_split[split_name] += 1

if dup_per_split:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(dup_per_split.keys(), dup_per_split.values(), color="salmon", edgecolor="black")
    ax.set_title("Duplicate Files to Remove per Split")
    ax.set_ylabel("Count")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{REPORTS_DIR}/duplicates_per_split.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# FER+ label analysis (dry-run only, no relabeling)

fer2013_csv_path = Path(f"{DRIVE_ROOT}/data/raw/fer2013/fer2013.csv")
ferplus_csv_path = Path(f"{DRIVE_ROOT}/data/raw/ferplus/fer2013new.csv")

fer_report = {
    "fer2013_csv": str(fer2013_csv_path),
    "ferplus_csv": str(ferplus_csv_path),
    "alignment_verified": False,
    "total_rows": 0,
    "mappable_rows": 0,
    "low_confidence_rows": 0,
    "excluded_surprise_rows": 0,
    "class_distribution": {},
    "confidence_scores": [],
}

fer_analysis_ok = False

if not fer2013_csv_path.exists() or not ferplus_csv_path.exists():
    print("WARNING: FER2013 or FER+ CSV not found. Skipping FER+ analysis.")
    if not fer2013_csv_path.exists():
        print(f"  Missing: {fer2013_csv_path}")
    if not ferplus_csv_path.exists():
        print(f"  Missing: {ferplus_csv_path}")
else:
    fer2013_df = pd.read_csv(fer2013_csv_path)
    ferplus_df = pd.read_csv(ferplus_csv_path)

    if len(fer2013_df) != len(ferplus_df):
        print(f"ERROR: Row count mismatch: fer2013={len(fer2013_df)}, ferplus={len(ferplus_df)}")
    else:
        fer_report["alignment_verified"] = True
        fer_report["total_rows"] = len(ferplus_df)

        available_cols = ferplus_df.columns.tolist()
        emotion_cols = [c for c in _FERPLUS_EMOTION_COLS if c in available_cols]

        class_dist = {c: 0 for c in CLASS_NAMES}
        low_confidence = 0
        excluded = 0
        confidence_scores = []

        for _, row in tqdm(ferplus_df.iterrows(), total=len(ferplus_df), desc="FER+ analysis"):
            votes = {col: row.get(col, 0) for col in emotion_cols}
            total_votes = sum(v for v in votes.values() if pd.notna(v))
            if total_votes == 0:
                low_confidence += 1
                continue

            class_votes = {c: 0.0 for c in CLASS_NAMES}
            for emo, count in votes.items():
                if pd.isna(count):
                    continue
                mapped_class = _FERPLUS_TO_3CLASS.get(emo)
                if mapped_class is None:
                    excluded += float(count)
                    continue
                class_votes[mapped_class] += float(count)

            total_mapped = sum(class_votes.values())
            if total_mapped == 0:
                excluded += 1
                continue

            winner = max(class_votes, key=lambda k: class_votes[k])
            confidence = class_votes[winner] / total_mapped
            confidence_scores.append(confidence)

            if confidence < FERPLUS_CONFIDENCE_THRESHOLD:
                low_confidence += 1
                continue

            class_dist[winner] += 1

        fer_report["mappable_rows"] = sum(class_dist.values())
        fer_report["low_confidence_rows"] = low_confidence
        fer_report["excluded_surprise_rows"] = int(excluded)
        fer_report["class_distribution"] = class_dist
        fer_report["confidence_scores"] = confidence_scores
        fer_analysis_ok = True

        print("\n--- FER+ Label Analysis ---")
        print(f"Total rows         : {fer_report['total_rows']:,}")
        print(f"Alignment verified : {fer_report['alignment_verified']}")
        print(f"Mappable rows      : {fer_report['mappable_rows']:,}")
        print(f"Low confidence     : {fer_report['low_confidence_rows']}")
        print(f"Excluded (surprise): {fer_report['excluded_surprise_rows']}")
        print(f"Class distribution : {class_dist}")

# Save report (without large confidence_scores list)
fer_report_save = {k: v for k, v in fer_report.items() if k != "confidence_scores"}
fer_report_path = Path(REPORTS_DIR) / "fer_label_analysis.json"
fer_report_path.write_text(json.dumps(fer_report_save, indent=2, ensure_ascii=False))
print(f"Report saved to {fer_report_path}")

In [ ]:
# FER+ results visualization

if fer_analysis_ok:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Bar chart of FER+ class distribution
    class_dist = fer_report["class_distribution"]
    colors = ["#ff6b6b", "#4ecdc4", "#45b7d1"]
    ax1.bar(class_dist.keys(), class_dist.values(), color=colors, edgecolor="black")
    ax1.set_title("FER+ 3-Class Distribution (confidence >= {:.0%})".format(FERPLUS_CONFIDENCE_THRESHOLD))
    ax1.set_ylabel("Count")
    ax1.grid(axis="y", alpha=0.3)
    for i, (cls, cnt) in enumerate(class_dist.items()):
        ax1.text(i, cnt + 50, f"{cnt:,}", ha="center", fontsize=10)

    # Confidence distribution histogram
    confidence_scores = fer_report["confidence_scores"]
    ax2.hist(confidence_scores, bins=50, color="mediumpurple", edgecolor="black", alpha=0.7)
    ax2.axvline(FERPLUS_CONFIDENCE_THRESHOLD, color="red", linestyle="--",
                label=f"Threshold: {FERPLUS_CONFIDENCE_THRESHOLD}")
    ax2.set_title("FER+ Confidence Score Distribution")
    ax2.set_xlabel("Confidence")
    ax2.set_ylabel("Count")
    ax2.legend()
    ax2.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{REPORTS_DIR}/ferplus_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Summary table
    summary_data = {
        "Metric": ["Total Rows", "Mappable (conf >= threshold)", "Low Confidence", "Excluded (surprise)"],
        "Value": [
            f"{fer_report['total_rows']:,}",
            f"{fer_report['mappable_rows']:,}",
            f"{fer_report['low_confidence_rows']}",
            f"{fer_report['excluded_surprise_rows']}",
        ],
    }
    display(pd.DataFrame(summary_data).style.set_caption("FER+ Analysis Summary"))
else:
    print("FER+ analysis was skipped (CSV files not available).")

In [ ]:
# Cleaning summary table + confirmation

# Collect all files to remove
corrupt_files = set()
for paths in issues.values():
    corrupt_files.update(paths)

duplicate_files = set()
for group in all_dup_groups:
    for rel_path in group[1:]:  # keep first of each group
        duplicate_files.add(rel_path)

# Some files may appear in both lists
overlap = corrupt_files & duplicate_files
total_to_remove = len(corrupt_files | duplicate_files)

summary_rows = [
    {"Category": "Corrupt (unreadable)", "Count": len(issues["unreadable"]),
     "Percentage": f"{len(issues['unreadable'])/len(all_images_for_scan)*100:.2f}%"},
    {"Category": "Corrupt (too small file)", "Count": len(issues["too_small_file"]),
     "Percentage": f"{len(issues['too_small_file'])/len(all_images_for_scan)*100:.2f}%"},
    {"Category": "Corrupt (too small image)", "Count": len(issues["too_small_image"]),
     "Percentage": f"{len(issues['too_small_image'])/len(all_images_for_scan)*100:.2f}%"},
    {"Category": "Corrupt (low variance)", "Count": len(issues["low_variance"]),
     "Percentage": f"{len(issues['low_variance'])/len(all_images_for_scan)*100:.2f}%"},
    {"Category": "Exact duplicates", "Count": dup_report["exact_duplicate_files"],
     "Percentage": f"{dup_report['exact_duplicate_files']/len(all_images_for_scan)*100:.2f}%"},
    {"Category": "Near duplicates", "Count": dup_report["near_duplicate_files"],
     "Percentage": f"{dup_report['near_duplicate_files']/len(all_images_for_scan)*100:.2f}%"},
    {"Category": "Overlap (both)", "Count": len(overlap),
     "Percentage": f"{len(overlap)/len(all_images_for_scan)*100:.2f}%"},
    {"Category": "TOTAL TO REMOVE", "Count": total_to_remove,
     "Percentage": f"{total_to_remove/len(all_images_for_scan)*100:.2f}%"},
]

df_summary = pd.DataFrame(summary_rows)
display(df_summary.style.set_caption("Cleaning Summary"))

print(f"\nTotal images to remove: {total_to_remove:,} / {len(all_images_for_scan):,}")
print(f"  Corrupt  -> quarantine/corrupt/")
print(f"  Duplicates -> quarantine/duplicates/")
print(f"  Remaining: {len(all_images_for_scan) - total_to_remove:,} images")

confirm = input("\nProceed with cleaning? (yes/no): ").strip().lower()
if confirm != "yes":
    raise RuntimeError("Cleaning aborted by user.")
print("Cleaning confirmed. Proceeding...")

In [ ]:
# Apply cleaning: move corrupt and duplicate images to quarantine
import shutil

quarantine_corrupt_dir = Path(QUARANTINE_DIR) / "corrupt"
quarantine_dups_dir = Path(QUARANTINE_DIR) / "duplicates"

moved_corrupt = 0
moved_dups = 0

# Move corrupt images
print("Moving corrupt images to quarantine...")
for rel_path in tqdm(sorted(corrupt_files), desc="Quarantine corrupt"):
    src = data_root / rel_path
    if not src.exists():
        continue
    dest = quarantine_corrupt_dir / rel_path
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(src), str(dest))
    moved_corrupt += 1

# Move duplicate images (skip if already moved as corrupt)
print("Moving duplicate images to quarantine...")
for rel_path in tqdm(sorted(duplicate_files), desc="Quarantine duplicates"):
    src = data_root / rel_path
    if not src.exists():
        continue
    dest = quarantine_dups_dir / rel_path
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(src), str(dest))
    moved_dups += 1

# Recount remaining images
remaining = sum(
    1 for p in data_root.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)

print(f"\n=== Cleaning Complete ===")
print(f"Moved corrupt    : {moved_corrupt}")
print(f"Moved duplicates : {moved_dups}")
print(f"Total moved      : {moved_corrupt + moved_dups}")
print(f"Remaining images : {remaining:,}")

In [ ]:
# Stratified split: pool train+val, split into 80/10/10
from sklearn.model_selection import train_test_split

cleaned_root = Path(CLEANED_DIR)

# Collect all images from train/ and val/ in processed_merged
all_files = []
all_labels = []

for split_name in ["train", "val"]:
    split_dir = data_root / split_name
    if not split_dir.exists():
        print(f"Warning: {split_dir} does not exist, skipping.")
        continue
    for class_name in CLASS_NAMES:
        class_dir = split_dir / class_name
        if not class_dir.exists():
            continue
        for img_path in class_dir.iterdir():
            if img_path.is_file() and img_path.suffix.lower() in IMAGE_EXTENSIONS:
                all_files.append(img_path)
                all_labels.append(class_name)

# Also include test/ images if they exist
test_split_dir = data_root / "test"
if test_split_dir.exists():
    for class_name in CLASS_NAMES:
        class_dir = test_split_dir / class_name
        if not class_dir.exists():
            continue
        for img_path in class_dir.iterdir():
            if img_path.is_file() and img_path.suffix.lower() in IMAGE_EXTENSIONS:
                all_files.append(img_path)
                all_labels.append(class_name)

print(f"Total images pooled: {len(all_files):,}")
for cls in CLASS_NAMES:
    print(f"  {cls}: {all_labels.count(cls):,}")

# Stratified split: first separate test set (10%)
train_val_files, test_files, train_val_labels, test_labels = train_test_split(
    all_files, all_labels,
    test_size=MERGED_TEST_RATIO,
    random_state=RANDOM_SEED,
    stratify=all_labels,
)

# Then separate val from remaining (10% of total = ~11.1% of remaining)
val_ratio_adjusted = MERGED_VAL_RATIO / (1.0 - MERGED_TEST_RATIO)
train_files, val_files, train_labels, val_labels = train_test_split(
    train_val_files, train_val_labels,
    test_size=val_ratio_adjusted,
    random_state=RANDOM_SEED,
    stratify=train_val_labels,
)

# Copy files to cleaned directory
print("\nCopying files to cleaned directory...")
split_summary = {}

for split_name, files, labels in [
    ("train", train_files, train_labels),
    ("val", val_files, val_labels),
    ("test", test_files, test_labels),
]:
    split_counts = {c: 0 for c in CLASS_NAMES}
    for file_path, label in tqdm(
        zip(files, labels), total=len(files), desc=f"Copy {split_name}"
    ):
        dest_dir = cleaned_root / split_name / label
        dest_dir.mkdir(parents=True, exist_ok=True)
        dest_path = dest_dir / file_path.name

        # Handle filename collisions
        if dest_path.exists():
            stem = file_path.stem
            suffix = file_path.suffix
            counter = 1
            while dest_path.exists():
                dest_path = dest_dir / f"{stem}_{counter}{suffix}"
                counter += 1

        shutil.copy2(str(file_path), str(dest_path))
        split_counts[label] += 1

    split_summary[split_name] = split_counts
    total = sum(split_counts.values())
    print(f"\n{split_name}: {total:,} images")
    for cls in CLASS_NAMES:
        print(f"  {cls}: {split_counts[cls]:,}")

print(f"\nOutput directory: {cleaned_root}")

In [ ]:
# Final dataset report (cleaned data)

cleaned_root = Path(CLEANED_DIR)
final_rows = []

for split in ["train", "val", "test"]:
    split_dir = cleaned_root / split
    counts = {}
    for cls in CLASS_NAMES:
        cls_dir = split_dir / cls
        if cls_dir.exists():
            counts[cls] = sum(
                1 for f in cls_dir.iterdir()
                if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
            )
        else:
            counts[cls] = 0
    final_rows.append({"split": split, **counts, "total": sum(counts.values())})

# Add totals
totals = {"split": "TOTAL"}
for cls in CLASS_NAMES:
    totals[cls] = sum(r[cls] for r in final_rows)
totals["total"] = sum(r["total"] for r in final_rows)
final_rows.append(totals)

df_final = pd.DataFrame(final_rows).set_index("split")
print("=== Cleaned Dataset Counts ===")
display(df_final.style.format("{:,}").set_caption("Cleaned Dataset (data/cleaned/)"))

# Bar chart of final class distribution
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(CLASS_NAMES))
width = 0.25
colors_split = ["#2196F3", "#FF9800", "#4CAF50"]

for i, split in enumerate(["train", "val", "test"]):
    row = df_final.loc[split]
    values = [row[c] for c in CLASS_NAMES]
    ax.bar(x + i * width, values, width, label=split, color=colors_split[i])

ax.set_xlabel("Class")
ax.set_ylabel("Count")
ax.set_title("Cleaned Dataset: Class Distribution by Split")
ax.set_xticks(x + width)
ax.set_xticklabels(CLASS_NAMES)
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(f"{REPORTS_DIR}/cleaned_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Before/after comparison

# Original counts from initial scan
original_per_class = {}
for cls in CLASS_NAMES:
    original_per_class[cls] = df_counts.loc["TOTAL", cls] if "TOTAL" in df_counts.index else 0

# Cleaned counts
cleaned_per_class = {}
for cls in CLASS_NAMES:
    cleaned_per_class[cls] = df_final.loc["TOTAL", cls] if "TOTAL" in df_final.index else 0

# Side-by-side bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(CLASS_NAMES))
width = 0.35

orig_values = [original_per_class[c] for c in CLASS_NAMES]
clean_values = [cleaned_per_class[c] for c in CLASS_NAMES]

ax1.bar(x - width/2, orig_values, width, label="Original", color="#FF7043")
ax1.bar(x + width/2, clean_values, width, label="Cleaned", color="#66BB6A")
ax1.set_xlabel("Class")
ax1.set_ylabel("Count")
ax1.set_title("Before vs After Cleaning")
ax1.set_xticks(x)
ax1.set_xticklabels(CLASS_NAMES)
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

# Summary table as text
orig_total = sum(orig_values)
clean_total = sum(clean_values)
removed_total = orig_total - clean_total
pct_removed = (removed_total / orig_total * 100) if orig_total > 0 else 0

table_text = (
    f"Original : {orig_total:>8,}\n"
    f"Cleaned  : {clean_total:>8,}\n"
    f"Removed  : {removed_total:>8,}\n"
    f"Pct removed: {pct_removed:.1f}%"
)
ax2.text(0.5, 0.5, table_text, transform=ax2.transAxes, fontsize=14,
         verticalalignment="center", horizontalalignment="center",
         fontfamily="monospace",
         bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
ax2.set_title("Summary")
ax2.axis("off")

plt.tight_layout()
plt.savefig(f"{REPORTS_DIR}/before_after.png", dpi=150, bbox_inches="tight")
plt.show()

# Detailed per-class table
comparison_rows = []
for cls in CLASS_NAMES:
    orig = original_per_class[cls]
    clean = cleaned_per_class[cls]
    removed = orig - clean
    pct = (removed / orig * 100) if orig > 0 else 0
    comparison_rows.append({
        "Class": cls,
        "Original": f"{orig:,}",
        "Cleaned": f"{clean:,}",
        "Removed": f"{removed:,}",
        "% Removed": f"{pct:.1f}%",
    })
comparison_rows.append({
    "Class": "TOTAL",
    "Original": f"{orig_total:,}",
    "Cleaned": f"{clean_total:,}",
    "Removed": f"{removed_total:,}",
    "% Removed": f"{pct_removed:.1f}%",
})

display(pd.DataFrame(comparison_rows).style.set_caption("Before/After Comparison"))

In [ ]:
# Augmentation preview Phase 1 (light)
import albumentations as A

# Pick a sample image from cleaned/train/
sample_class = CLASS_NAMES[0]
sample_dir = cleaned_root / "train" / sample_class
sample_images_list = [
    f for f in sample_dir.iterdir()
    if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
]
sample_img_path = sample_images_list[0]
sample_img = cv2.imread(str(sample_img_path))
sample_img = cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB)

# Phase 1 transforms (without Normalize and ToTensorV2 for visualization)
phase1_viz = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=10, p=0.3),
    A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.0, hue=0.0, p=0.3),
])

fig, axes = plt.subplots(1, 7, figsize=(21, 3))
fig.suptitle("Phase 1 \u2014 Light Augmentation", fontsize=14)

# Original (resized)
resized = A.Resize(IMAGE_SIZE, IMAGE_SIZE)(image=sample_img)["image"]
axes[0].imshow(resized)
axes[0].set_title("Original", fontsize=10)
axes[0].axis("off")

# 6 augmented versions
for i in range(1, 7):
    augmented = phase1_viz(image=sample_img)["image"]
    axes[i].imshow(augmented)
    axes[i].set_title(f"Aug {i}", fontsize=10)
    axes[i].axis("off")

plt.tight_layout()
plt.savefig(f"{REPORTS_DIR}/phase1_augmentation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Augmentation preview Phase 2 (strong)

# Phase 2 transforms (without Normalize and ToTensorV2 for visualization)
phase2_viz = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.OneOf([
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8)),
    ], p=0.5),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5)),
        A.GaussNoise(std_range=(0.02, 0.1)),
    ], p=0.3),
    A.OneOf([
        A.GridDistortion(num_steps=5, distort_limit=0.1),
        A.OpticalDistortion(distort_limit=0.05),
    ], p=0.2),
    A.CoarseDropout(
        num_holes_range=(1, 1),
        hole_height_range=(int(IMAGE_SIZE * 0.05), int(IMAGE_SIZE * 0.25)),
        hole_width_range=(int(IMAGE_SIZE * 0.05), int(IMAGE_SIZE * 0.25)),
        fill=0,
        p=0.3,
    ),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
])

fig, axes = plt.subplots(1, 7, figsize=(21, 3))
fig.suptitle("Phase 2 \u2014 Strong Augmentation", fontsize=14)

# Original (resized)
axes[0].imshow(resized)
axes[0].set_title("Original", fontsize=10)
axes[0].axis("off")

# 6 augmented versions
for i in range(1, 7):
    augmented = phase2_viz(image=sample_img)["image"]
    axes[i].imshow(augmented)
    axes[i].set_title(f"Aug {i}", fontsize=10)
    axes[i].axis("off")

plt.tight_layout()
plt.savefig(f"{REPORTS_DIR}/phase2_augmentation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# MixUp/CutMix preview

# Pick 2 images from different classes
img_a_dir = cleaned_root / "train" / CLASS_NAMES[0]
img_b_dir = cleaned_root / "train" / CLASS_NAMES[2]

img_a_path = next(
    f for f in img_a_dir.iterdir()
    if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
)
img_b_path = next(
    f for f in img_b_dir.iterdir()
    if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
)

img_a = cv2.imread(str(img_a_path))
img_a = cv2.cvtColor(img_a, cv2.COLOR_BGR2RGB)
img_a = cv2.resize(img_a, (IMAGE_SIZE, IMAGE_SIZE))

img_b = cv2.imread(str(img_b_path))
img_b = cv2.cvtColor(img_b, cv2.COLOR_BGR2RGB)
img_b = cv2.resize(img_b, (IMAGE_SIZE, IMAGE_SIZE))

# MixUp: linear interpolation
np.random.seed(RANDOM_SEED)
lam_mixup = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
mixed_mixup = (lam_mixup * img_a.astype(np.float32) + (1 - lam_mixup) * img_b.astype(np.float32)).astype(np.uint8)

# CutMix: paste random patch
lam_cutmix = np.random.beta(CUTMIX_ALPHA, CUTMIX_ALPHA)
h, w = IMAGE_SIZE, IMAGE_SIZE
cut_ratio = (1.0 - lam_cutmix) ** 0.5
cut_w = int(w * cut_ratio)
cut_h = int(h * cut_ratio)
cx = np.random.randint(0, w)
cy = np.random.randint(0, h)
bbx1 = max(0, cx - cut_w // 2)
bby1 = max(0, cy - cut_h // 2)
bbx2 = min(w, cx + cut_w // 2)
bby2 = min(h, cy + cut_h // 2)

mixed_cutmix = img_a.copy()
mixed_cutmix[bby1:bby2, bbx1:bbx2] = img_b[bby1:bby2, bbx1:bbx2]
actual_lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1)) / (w * h)

# Draw bounding box overlay on cutmix result
mixed_cutmix_viz = mixed_cutmix.copy()
cv2.rectangle(mixed_cutmix_viz, (bbx1, bby1), (bbx2, bby2), (255, 0, 0), 2)

# Plot: 2 rows (MixUp, CutMix) x 3 cols (img_a, img_b, mixed)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.suptitle("MixUp & CutMix Preview", fontsize=14)

# MixUp row
axes[0, 0].imshow(img_a)
axes[0, 0].set_title(f"Image A ({CLASS_NAMES[0]})")
axes[0, 0].axis("off")

axes[0, 1].imshow(img_b)
axes[0, 1].set_title(f"Image B ({CLASS_NAMES[2]})")
axes[0, 1].axis("off")

axes[0, 2].imshow(mixed_mixup)
soft_label_mixup = f"[{lam_mixup:.2f}, 0, {1-lam_mixup:.2f}]"
axes[0, 2].set_title(f"MixUp (\u03bb={lam_mixup:.3f})\nSoft: {soft_label_mixup}")
axes[0, 2].axis("off")

# CutMix row
axes[1, 0].imshow(img_a)
axes[1, 0].set_title(f"Image A ({CLASS_NAMES[0]})")
axes[1, 0].axis("off")

axes[1, 1].imshow(img_b)
axes[1, 1].set_title(f"Image B ({CLASS_NAMES[2]})")
axes[1, 1].axis("off")

axes[1, 2].imshow(mixed_cutmix_viz)
soft_label_cutmix = f"[{actual_lam:.2f}, 0, {1-actual_lam:.2f}]"
axes[1, 2].set_title(f"CutMix (\u03bb={actual_lam:.3f})\nSoft: {soft_label_cutmix}")
axes[1, 2].axis("off")

plt.tight_layout()
plt.savefig(f"{REPORTS_DIR}/mixup_cutmix_preview.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Normalize before/after comparison + pixel distribution

# Take the same sample image
sample_resized = cv2.resize(sample_img, (IMAGE_SIZE, IMAGE_SIZE))

# Normalized version (using albumentations Normalize)
normalize_transform = A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
normalized = normalize_transform(image=sample_resized)["image"]

# Denormalize for display
mean = np.array(IMAGENET_MEAN)
std = np.array(IMAGENET_STD)
denormalized = (normalized * std + mean)
denormalized = np.clip(denormalized * 255, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Normalization: Before vs After", fontsize=14)

# Unnormalized image
axes[0, 0].imshow(sample_resized)
axes[0, 0].set_title("Unnormalized (Resized)")
axes[0, 0].axis("off")

# Normalized image (denormalized for display)
axes[0, 1].imshow(denormalized)
axes[0, 1].set_title("Normalized (denorm for display)")
axes[0, 1].axis("off")

# Pixel histograms: unnormalized
channel_names = ["Red", "Green", "Blue"]
channel_colors = ["red", "green", "blue"]
for c in range(3):
    axes[1, 0].hist(
        sample_resized[:, :, c].flatten(),
        bins=50, alpha=0.5, color=channel_colors[c], label=channel_names[c],
    )
axes[1, 0].set_title("Pixel Distribution (Unnormalized)")
axes[1, 0].set_xlabel("Pixel Value (0-255)")
axes[1, 0].set_ylabel("Count")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Pixel histograms: normalized
for c in range(3):
    axes[1, 1].hist(
        normalized[:, :, c].flatten(),
        bins=50, alpha=0.5, color=channel_colors[c], label=channel_names[c],
    )
axes[1, 1].set_title("Pixel Distribution (Normalized)")
axes[1, 1].set_xlabel("Normalized Value")
axes[1, 1].set_ylabel("Count")
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{REPORTS_DIR}/normalization_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Unnormalized - min: {sample_resized.min()}, max: {sample_resized.max()}, mean: {sample_resized.mean():.2f}")
print(f"Normalized   - min: {normalized.min():.3f}, max: {normalized.max():.3f}, mean: {normalized.mean():.3f}")
print(f"ImageNet mean: {IMAGENET_MEAN}")
print(f"ImageNet std : {IMAGENET_STD}")

# Ozet ve Sonraki Adimlar

## Yapilan Islemler
- **Bozuk goruntu tespiti**: Okunamayan, cok kucuk ve dusuk varyansli dosyalar tespit edildi ve `quarantine/corrupt/` klasorune tasindi
- **Kopya tespiti**: pHash (hash_size=16) ile tam ve yakin kopyalar (hamming <= 6) tespit edildi, `quarantine/duplicates/` klasorune tasindi
- **Cross-split leak kontrolu**: Train/val/test arasinda kopya sizintilari raporlandi
- **FER+ etiket analizi**: Crowd-vote guven skorlari ve 3-sinif eslesme dagilimi incelendi
- **Stratified bolum**: %80 train / %10 val / %10 test oraninda sinif dagilimi korunarak bolum yapildi
- **Augmentation onizleme**: Phase 1 (light) ve Phase 2 (strong) augmentation pipeline'lari gorsellestirildi
- **MixUp/CutMix onizleme**: Batch-level augmentation teknikleri gorsellestirildi
- **Normalizasyon**: ImageNet mean/std ile normalizasyonun piksel dagilimina etkisi incelendi

## Drive Yapi Ozeti
```
MyDrive/Tubitak-2209B/
  data/
    processed_merged/       # Kaynak veri (10 dataset, ~187K goruntu)
    cleaned/                # Temizlenmis veri (train/val/test)
      train/
        negative/
        neutral/
        positive/
      val/
      test/
    quarantine/             # Kaldirilan goruntuler
      corrupt/
      duplicates/
  results/data_cleaning/    # Raporlar (JSON) ve grafikler (PNG)
```

## Sonraki Notebook
- **`02_model_training.ipynb`**: Temizlenmis veri uzerinde EfficientNet-B3 model egitimi
  - Phase 1: Head-only training (5 epoch, lr=1e-3)
  - Phase 2: Full fine-tuning (25 epoch, lr=1e-4)
  - Focal Loss + MixUp/CutMix
  - Early stopping (patience=7)